# UC13 Pipeline — End-to-End Test Notebook

Runs each production script's `main()` in sequence, exactly as the Databricks Workflow does.
Intermediate verification queries between steps let you inspect state before continuing.

**Execution order:**
```
[ 1. Config ]  →  [ 2. Setup VS ]  →  [ 3. Download/Upload ]
  →  [ 4. Verify upload_log ]  →  [ 5. Classifier ]
  →  [ 6. Verify doc_relevance ]  →  [ 7. Parser ]
  →  [ 8. Verify chunks/embeddings ]  →  [ 9. Profiler ]
  →  [ 10. Verify profile ]
```

**Before running:**
- Confirm the repo is cloned under `/Workspace/Repos/`
- Confirm secrets scope `uc13` is populated (see `workflows/README.md`)
- Set `SP_COMPANY_NAME` in the widget below

In [ ]:
# ── Cell 1: Config ────────────────────────────────────────────────────────────
# Edit SP_COMPANY_NAME to switch companies. Everything else uses safe defaults.

import os, sys
from pathlib import Path

# Widgets so the Workflow UI can also inject values.
dbutils.widgets.text("sp_company_name",     "Elder Care")
dbutils.widgets.text("catalog",             "uc13")
dbutils.widgets.text("schema",              "ingestion")
dbutils.widgets.text("embedding_endpoint",  "databricks-bge-large-en")
dbutils.widgets.text("llm_endpoint",        "databricks-meta-llama-3-3-70b-instruct")

# ── Resolve repo root from this notebook's path ───────────────────────────────
def get_current_path():
    try:
        nb_path = (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .notebookPath().get()
        )
        return Path("/Workspace") / nb_path.lstrip("/")
    except Exception:
        return Path(os.getcwd())

def find_repo_root(marker="agents"):
    p = get_current_path()
    if p.is_file():
        p = p.parent
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return str(candidate)
    raise RuntimeError(f"Could not find a parent directory containing '{marker}'")

REPO_ROOT = find_repo_root()
SCRIPTS   = str(Path(REPO_ROOT) / "jobs" / "scripts")

for p in [REPO_ROOT, SCRIPTS]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"repo_root : {REPO_ROOT}")
print(f"scripts   : {SCRIPTS}")
print(f"company   : {dbutils.widgets.get('sp_company_name')}")

---
## Step 0 — Setup Vector Search
Run this cell **once** per workspace. Skip it on subsequent runs — it is idempotent but slow.

In [ ]:
# ── Cell 2: Setup Vector Search (run once) ────────────────────────────────────
# Comment out after the first successful run.

import importlib, setup_vector_search as s0
importlib.reload(s0)
s0.main()

---
## Step 1 — Download & Upload (Phase 1)
Downloads from SharePoint → local scratch → UC Volume.  
Writes `uc13.ingestion.upload_log` with Priority Tier signals.

In [ ]:
# ── Cell 3: Download & Upload ─────────────────────────────────────────────────
import importlib, download_upload as s1
importlib.reload(s1)
s1.main()

In [ ]:
# ── Cell 4: Verify upload_log ─────────────────────────────────────────────────
catalog      = dbutils.widgets.get("catalog")
schema       = dbutils.widgets.get("schema")
company_name = dbutils.widgets.get("sp_company_name")

display(
    spark.sql(f"""
        SELECT
            priority_tier,
            format,
            COUNT(*) AS files,
            SUM(size_bytes) / 1e6 AS total_mb
        FROM {catalog}.{schema}.upload_log
        WHERE company_name = '{company_name}'
        GROUP BY priority_tier, format
        ORDER BY priority_tier DESC, files DESC
    """)
)

# Spot-check: which files got Priority Tier = True?
display(
    spark.sql(f"""
        SELECT file_name, folder_path, priority_reason, mod_date
        FROM {catalog}.{schema}.upload_log
        WHERE company_name = '{company_name}'
          AND priority_tier = true
        ORDER BY folder_path
    """)
)

---
## Step 2 — Document Classifier (Phase 2a)
Calls the LLM in batches of 20 to assign workstream tags + priority tier.  
Writes `uc13.classification.doc_relevance`.

In [ ]:
# ── Cell 5: Document Classifier ───────────────────────────────────────────────
import importlib, document_classifier as s2
importlib.reload(s2)
s2.main()

In [ ]:
# ── Cell 6: Verify doc_relevance ──────────────────────────────────────────────
company_name = dbutils.widgets.get("sp_company_name")

# Distribution by workstream tag
display(
    spark.sql(f"""
        SELECT
            explode(workstream) AS tag,
            priority_tier,
            COUNT(*) AS files
        FROM uc13.classification.doc_relevance
        WHERE company_name = '{company_name}'
        GROUP BY tag, priority_tier
        ORDER BY files DESC
    """)
)

# Files flagged for parsing
display(
    spark.sql(f"""
        SELECT
            should_parse,
            priority_tier,
            extraction_confidence,
            COUNT(*) AS files
        FROM uc13.classification.doc_relevance
        WHERE company_name = '{company_name}'
        GROUP BY should_parse, priority_tier, extraction_confidence
        ORDER BY should_parse DESC, priority_tier DESC
    """)
)

# Spot-check: verify Priority Tier = True files look correct
display(
    spark.sql(f"""
        SELECT filename, folder_path, workstream, priority_reason, extraction_confidence
        FROM uc13.classification.doc_relevance
        WHERE company_name = '{company_name}'
          AND priority_tier = true
        ORDER BY folder_path
    """)
)

# Red flag: should_parse=true with null/empty workstream
suspicious = spark.sql(f"""
    SELECT filename, folder_path, workstream
    FROM uc13.classification.doc_relevance
    WHERE company_name = '{company_name}'
      AND should_parse = true
      AND (workstream IS NULL OR size(workstream) = 0)
""").count()
print(f"\n⚠️  Files with should_parse=true but no workstream tag: {suspicious}")

---
## Step 3 — Ingestion Parser (Phase 2b)
Parses `should_parse=true` files into chunks, generates embeddings.  
Priority Tier files are processed first.  
Writes `uc13.ingestion.chunks` and `uc13.ingestion.embeddings`.

In [ ]:
# ── Cell 7: Ingestion Parser ──────────────────────────────────────────────────
import importlib, ingestion_parser as s3
importlib.reload(s3)
s3.main()

In [ ]:
# ── Cell 8: Verify chunks + embeddings ────────────────────────────────────────
company_name = dbutils.widgets.get("sp_company_name")

# Chunk stats by file type
display(
    spark.sql(f"""
        SELECT
            file_type,
            COUNT(DISTINCT doc_id)  AS documents,
            COUNT(*)                AS total_chunks,
            ROUND(AVG(char_count))  AS avg_chars,
            MIN(char_count)         AS min_chars,
            MAX(char_count)         AS max_chars
        FROM uc13.ingestion.chunks
        WHERE company_name = '{company_name}'
        GROUP BY file_type
        ORDER BY file_type
    """)
)

# Embeddings: confirm workstream + priority_tier columns are populated
display(
    spark.sql(f"""
        SELECT
            workstream,
            priority_tier,
            COUNT(*) AS embeddings
        FROM uc13.ingestion.embeddings
        WHERE company_name = '{company_name}'
        GROUP BY workstream, priority_tier
        ORDER BY priority_tier DESC
    """)
)

# Red flag: chunks with no section_header (carry-forward fix should eliminate these)
headerless = spark.sql(f"""
    SELECT file_name, chunk_index, LEFT(chunk_text, 120) AS preview
    FROM uc13.ingestion.chunks
    WHERE company_name = '{company_name}'
      AND section_header IS NULL
    LIMIT 10
""").count()
print(f"\n⚠️  Chunks with null section_header: {headerless}")

# Red flag: oversized Excel chunks that could truncate in BGE
display(
    spark.sql(f"""
        SELECT file_name, section_header, char_count
        FROM uc13.ingestion.chunks
        WHERE company_name = '{company_name}'
          AND file_type = 'xlsx' AND char_count > 32000
        ORDER BY char_count DESC
        LIMIT 10
    """)
)

In [ ]:
# ── Cell 8b: Smoke-test semantic search ───────────────────────────────────────
# Quick retrieval check before running the profiler.

from agents.shared.retrieval import semantic_search

results = semantic_search(
    query="company overview what does this company do business description",
    spark=spark,
    top_k=5,
    file_name_filter=["CIM", "Business", "Overview"],
    workstream_filter=["BUSINESS_MODEL"],
    min_chunk_length=100,
)

print(f"\nReturned {len(results)} chunks\n")
for r in results:
    print(f"  [{r.file_name}]  section: {r.section_header}")
    print(f"  {r.chunk_text[:200]}")
    print()

---
## Step 4 — Company Profiler (Phase 2b)
Retrieves context for each profiling dimension, calls the LLM, and saves a structured profile.  
`company_name` always comes from the widget — never from LLM output.

In [ ]:
# ── Cell 9: Company Profiler ──────────────────────────────────────────────────
import importlib, company_profiler as s4
importlib.reload(s4)
s4.main()

In [ ]:
# ── Cell 10: Verify company_profile ───────────────────────────────────────────
company_name = dbutils.widgets.get("sp_company_name")

display(
    spark.sql("""
        SELECT
            company_name,
            industry_overlay,
            overlay_confidence,
            revenue_model,
            deal_type,
            banked,
            vertical_subsector,
            data_room_gaps,
            created_at
        FROM uc13.classification.company_profile
        ORDER BY created_at DESC
        LIMIT 5
    """)
)

row = spark.sql(f"""
    SELECT * FROM uc13.classification.company_profile
    WHERE company_name = '{company_name}'
    ORDER BY created_at DESC LIMIT 1
""").collect()[0]

print("\n=== Latest profile ===")
print(f"  company_name          : {row.company_name}")
print(f"  industry_overlay      : {row.industry_overlay} ({row.overlay_confidence})")
print(f"  revenue_model         : {row.revenue_model}")
print(f"  revenue_model_note    : {row.revenue_model_note}")
print(f"  business_description  : {row.business_description}")
print(f"  company_size_indicators: {row.company_size_indicators}")
print(f"  deal_type             : {row.deal_type}")
print(f"  banked                : {row.banked}")
print(f"  banked_note           : {row.banked_note}")
print(f"  vertical_subsector    : {row.vertical_subsector}")
if row.data_room_gaps:
    print(f"  data_room_gaps ({len(row.data_room_gaps)})  :")
    for gap in row.data_room_gaps:
        print(f"    ! {gap}")
else:
    print("  data_room_gaps        : none")

---
## Utilities — Reset / Re-run a single step
Use these cells to truncate a table and re-run a specific step without starting over.

In [ ]:
# ── Cell 11: Reset helpers (run individually as needed) ───────────────────────
# Deletes only the current company's rows — other companies are preserved.
# Uncomment the table(s) you want to reset, run the cell, then re-run the step.

company_name = dbutils.widgets.get("sp_company_name")

# spark.sql(f"DELETE FROM uc13.ingestion.upload_log           WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.classification.doc_relevance   WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.ingestion.chunks               WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.ingestion.embeddings           WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.classification.company_profile WHERE company_name = '{company_name}'")

print(f"Uncomment the table(s) you want to reset for company='{company_name}', then run this cell.")

In [ ]:
# ── Cell 12: Full pipeline state summary ──────────────────────────────────────
# Shows total rows per table and rows for the current company widget value.

company_name = dbutils.widgets.get("sp_company_name")

tables = {
    "upload_log":      ("uc13.ingestion.upload_log",            "company_name"),
    "doc_relevance":   ("uc13.classification.doc_relevance",    "company_name"),
    "chunks":          ("uc13.ingestion.chunks",                "company_name"),
    "embeddings":      ("uc13.ingestion.embeddings",            "company_name"),
    "company_profile": ("uc13.classification.company_profile",  "company_name"),
}

print(f"{'Table':<25} {'Total':>8}  {'This company':>14}")
print("-" * 52)
for label, (fqt, col) in tables.items():
    try:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fqt}").collect()[0]["n"]
        mine  = spark.sql(f"SELECT COUNT(*) AS n FROM {fqt} WHERE {col} = '{company_name}'").collect()[0]["n"]
        print(f"  {label:<23} {total:>8,}  {mine:>14,}")
    except Exception as e:
        print(f"  {label:<23} {'(not found)':>8}")